<a href="https://colab.research.google.com/github/ynam0327-afk/REDRED/blob/main/domain_module.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
"""
2주차 - 도메인 구조 분석 모듈

"""

import re
import ast
import difflib
import pandas as pd
from google.colab import drive

drive.mount('/content/drive')
# =====================================================
# 경로 설정 — 본인 Drive 경로에 맞게 수정
# =====================================================
SMS_PATH = "/content/drive/MyDrive/REDRED/sms_url_enriched.csv"
FIRE_EVENTS_PATH = "/content/drive/MyDrive/REDRED/fire_events.csv"

SMS_OUT_PATH = "/content/drive/MyDrive/REDRED/sms_url_risk_analyzed.csv"
FIRE_OUT_PATH = "/content/drive/MyDrive/REDRED/fire_events_region_normalized.csv"


# =====================================================
# 1. 행정구역 약칭 -> 정식명칭 매핑
# =====================================================
SIDO_ABBR_TO_OFFICIAL = {
    "서울": "서울특별시", "부산": "부산광역시", "대구": "대구광역시",
    "인천": "인천광역시", "광주": "광주광역시", "대전": "대전광역시",
    "울산": "울산광역시", "세종": "세종특별자치시", "경기": "경기도",
    "강원": "강원특별자치도", "충북": "충청북도", "충남": "충청남도",
    "전북": "전북특별자치도", "전남": "전라남도", "경북": "경상북도",
    "경남": "경상남도", "제주": "제주특별자치도",
}

# 소방청 원문에서 시도 자리에 시/군 이름이 직접 오는 특례시 표기
SPECIAL_SIDO_ALIASES = {"창원": "경상남도"}

SIDO_SUFFIXES = ["특별자치시", "특별자치도", "광역시", "특별시", "도", "시"]


def _strip_sido_suffix(name: str) -> str:
    for suf in SIDO_SUFFIXES:
        if name.endswith(suf) and len(name) > len(suf):
            return name[: -len(suf)]
    return name


def normalize_sido(sido_raw):
    if not isinstance(sido_raw, str) or not sido_raw.strip():
        return {"sido_official": None, "is_matched": False, "note": "missing"}

    key = sido_raw.strip()

    if key in SIDO_ABBR_TO_OFFICIAL:
        return {"sido_official": SIDO_ABBR_TO_OFFICIAL[key], "is_matched": True, "note": None}

    if key in SPECIAL_SIDO_ALIASES:
        return {
            "sido_official": SPECIAL_SIDO_ALIASES[key],
            "is_matched": True,
            "note": f"'{key}'는 특례시 표기 - sigungu를 '{key}시'로 보정 필요",
        }

    # 접미사 뗀 형태로 재시도 ("세종시"->"세종", "경기도"->"경기", "광주광역시"->"광주")
    stripped = _strip_sido_suffix(key)
    if stripped in SIDO_ABBR_TO_OFFICIAL:
        return {
            "sido_official": SIDO_ABBR_TO_OFFICIAL[stripped],
            "is_matched": True,
            "note": f"원문이 정식/접미사 표기('{key}')라 약칭 매핑 대신 접미사 제거로 처리됨",
        }
    if stripped in SPECIAL_SIDO_ALIASES:
        return {
            "sido_official": SPECIAL_SIDO_ALIASES[stripped],
            "is_matched": True,
            "note": f"'{key}'는 특례시 표기 - sigungu를 '{stripped}시'로 보정 필요",
        }

    return {"sido_official": None, "is_matched": False, "note": f"미등록 시도명: {key}"}


def normalize_fire_events_region(events_df):
    out = events_df.copy()
    normalized = out["sido"].apply(normalize_sido)

    out["sido_official"] = normalized.apply(lambda d: d["sido_official"])
    out["sido_matched"] = normalized.apply(lambda d: d["is_matched"])
    out["region_note"] = normalized.apply(lambda d: d["note"])

    # '창원' 또는 '창원시'처럼 시도 자리에 도시명이 직접 온 특례시 케이스는
    # sigungu를 'OO시'로 보정한다. 이미 접미사가 붙어있으면 중복으로 붙이지 않는다.
    def fix_special_sigungu(row):
        raw = row["sido"]
        if not isinstance(raw, str):
            return row["sigungu"]
        stripped = _strip_sido_suffix(raw)
        if raw in SPECIAL_SIDO_ALIASES or stripped in SPECIAL_SIDO_ALIASES:
            city = stripped if stripped in SPECIAL_SIDO_ALIASES else raw
            return city if city.endswith("시") else city + "시"
        return row["sigungu"]

    special_row_mask = out["sido"].apply(
        lambda s: isinstance(s, str) and (s in SPECIAL_SIDO_ALIASES or _strip_sido_suffix(s) in SPECIAL_SIDO_ALIASES)
    )
    out.loc[special_row_mask, "sigungu"] = out.loc[special_row_mask].apply(fix_special_sigungu, axis=1)

    return out


# =====================================================
# 2. URL 도메인 신뢰도 분석
# =====================================================
OFFICIAL_DOMAINS = {
    "safekorea.go.kr",
    "korea119.go.kr",
    "nfa.go.kr",
    "weather.go.kr",
    "kma.go.kr",
    "vo.la",     # 홍수통제소, 지자체 등
    "url.kr",    # 은평구
}

# 대한민국 정부기관 공통 TLD — 개별 등록 없이 자동으로 공식 인정
OFFICIAL_TLD_SUFFIXES = (".go.kr",)

SHORTENER_DOMAINS = {
    "vo.la", "url.kr", "bit.ly", "han.gl", "t.ly", "goo.gl", "tinyurl.com",
    "t.co", "ow.ly", "is.gd", "buff.ly", "cutt.ly", "rb.gy",
}

RISKY_TLDS = {".tk", ".ml", ".ga", ".cf", ".gq", ".xyz", ".top", ".buzz"}


def is_official_domain(domain):
    if any(domain == d or domain.endswith("." + d) for d in OFFICIAL_DOMAINS):
        return True
    return any(domain.endswith(suffix) for suffix in OFFICIAL_TLD_SUFFIXES)


def typo_similarity_to_official(domain):
    best_score, best_domain = 0.0, None
    for official in OFFICIAL_DOMAINS:
        score = difflib.SequenceMatcher(None, domain, official).ratio()
        if score > best_score:
            best_score, best_domain = score, official
    return round(best_score, 3), best_domain


def domain_risk_score(domain):
    if not isinstance(domain, str) or not domain:
        return {}

    domain = domain.lower().strip()

    if is_official_domain(domain):
        return {
            "domain": domain, "is_official": True, "is_shortened": False,
            "has_risky_tld": False, "typo_similarity": 1.0, "typo_similar_to": domain,
            "contains_ip": False, "risk_score": 0.0,
        }

    is_shortened = domain in SHORTENER_DOMAINS
    has_risky_tld = any(domain.endswith(tld) for tld in RISKY_TLDS)
    contains_ip = bool(re.match(r'^\d{1,3}(\.\d{1,3}){3}$', domain))
    typo_sim, typo_target = typo_similarity_to_official(domain)

    risk = 0.0
    risk += 0.3 if is_shortened else 0
    risk += 0.2 if has_risky_tld else 0
    risk += 0.3 if contains_ip else 0
    risk += 0.2 * typo_sim

    return {
        "domain": domain, "is_official": False, "is_shortened": is_shortened,
        "has_risky_tld": has_risky_tld, "typo_similarity": typo_sim,
        "typo_similar_to": typo_target, "contains_ip": contains_ip,
        "risk_score": round(min(risk, 1.0), 3),
    }


def analyze_sms_url_domains(sms_df, domain_col="url_domains"):
    out = sms_df.copy()

    def evaluate_row(domains):
        if not isinstance(domains, list) or len(domains) == 0:
            return {"max_domain_risk": 0.0, "domain_risk_details": []}
        details = [domain_risk_score(d) for d in domains]
        max_risk = max(d["risk_score"] for d in details)
        return {"max_domain_risk": max_risk, "domain_risk_details": details}

    evaluated = out[domain_col].apply(evaluate_row)
    out["max_domain_risk"] = evaluated.apply(lambda d: d["max_domain_risk"])
    out["domain_risk_details"] = evaluated.apply(lambda d: d["domain_risk_details"])
    return out


# =====================================================
# 실행
# =====================================================
if __name__ == "__main__":
    # --- SMS URL 위험도 분석 ---
    sms = pd.read_csv(SMS_PATH)

    # url_domains는 CSV에 문자열("['vo.la']")로 저장되어 있으므로 리스트로 복원
    sms["url_domains"] = sms["url_domains"].apply(
        lambda x: ast.literal_eval(x) if isinstance(x, str) and x.startswith("[") else []
    )

    sms_analyzed = analyze_sms_url_domains(sms)
    sms_analyzed.to_csv(SMS_OUT_PATH, index=False, encoding="utf-8-sig")
    print(f"sms_url_risk_analyzed.csv 생성 완료 (URL 포함 {(sms_analyzed['max_domain_risk'].notna() & (sms['url_domains'].apply(len) > 0)).sum()}건 분석)")

    # --- 소방청 사건 지역 정규화 ---
    fire_events = pd.read_csv(FIRE_EVENTS_PATH)
    fire_normalized = normalize_fire_events_region(fire_events)
    fire_normalized.to_csv(FIRE_OUT_PATH, index=False, encoding="utf-8-sig")

    total = len(fire_normalized)
    matched = fire_normalized["sido_matched"].sum()
    print(f"fire_events_region_normalized.csv 생성 완료 (전체 {total}건 중 {matched}건 매핑 성공, {total - matched}건 확인 필요)")

    print("모든 작업 완료")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
sms_url_risk_analyzed.csv 생성 완료 (URL 포함 46건 분석)
fire_events_region_normalized.csv 생성 완료 (전체 17854건 중 17783건 매핑 성공, 71건 확인 필요)
모든 작업 완료
